# A1.15 · Overwhelming the human in the loop

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.14 · Repudiation and untraceability](https://spbreed.github.io/cyber-commons/lessons/A1.14.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Push approval volume up and measure the point at which review quality collapses.

**Why a security engineer needs it.** The approval gate is recorded as a control and operates as a click. At volume it approves everything, including the one request that mattered. The control it builds is: approval reserved for irreversible actions, with everything else bounded by policy (A3.6).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Approval is a genuine control at four requests a day. At four hundred it is a person clicking approve, and the control has quietly become a log of things somebody scrolled past.

> **At CyberTravels.** Approval on every refund is a real control at four a day. CyberTravels generates four hundred, and the control quietly becomes a log of things somebody scrolled past. R2.

## 2 · The framework

```
   requests/hour     4        40       400
   read carefully   yes      some      no
   approval is    control  friction  a log

   the control does not fail loudly. it degrades into a click.
```

**OWASP T10 — Overwhelming Human-in-the-Loop.**

Human approval is the control everyone reaches for first. It is placed at the
tool call — the right place — and it is genuinely strong for rare, consequential
decisions.

Then the system scales, and the arithmetic turns on it.

An agent generates approval requests at machine speed. A human reads them at
human speed. When the queue exceeds what a person can actually consider, the
behaviour does not degrade gracefully into "slower but careful". It degrades
into **approving without reading**, because the alternative is being the reason
nothing shipped.

The failure is invisible from inside the system. Every approval is recorded. The
audit trail shows a human decision on every action. The control appears to be
operating at 100%, and the thing being measured — that a human clicked — is not
the thing anyone cared about.

There is a second-order effect worth naming: an attacker who wants one approval
can *manufacture the volume that makes it likely*. Generate two hundred benign
requests, put the one that matters at position 173, and the control has been
defeated by arithmetic rather than by cleverness.

Approval is a control for irreversible actions. Used as a general-purpose gate
it becomes a click, and the risk register still counts it.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

Approval quality against volume, and the position an attacker chooses.

## 4 · The check, as a skill

Approval coverage at CyberTravels reads 100% at every volume, because coverage measures whether a human was asked. The skill models what reading does instead, and finds the volume at which the gate stops being one.

### The skill — [`skills/threats/approval-queue-saturation-model/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/approval-queue-saturation-model/SKILL.md)

```yaml
name: approval-queue-saturation-model
description: >-
  Model what happens to an approval gate as volume rises — coverage staying at
  100% while actual review collapses — and find the volume at which the control
  stops working. Use when reviewing human-in-the-loop design, approval fatigue,
  or a gate that has never been measured.
allowed-tools: Read, Grep, Glob
```

# Coverage stays at 100%; reading does not

An approval gate reports the same number at every volume, because coverage
measures whether a human was *asked*. What degrades is whether they read it,
and an attacker who can choose position only has to generate enough requests to
sit behind.

## When to use this

Any control whose enforcement is a person: approvals, exception reviews, alert
triage sign-off, change advisory.

## Procedure

**1 — Measure current volume,** per reviewer per day. Not the design volume —
the observed one, at peak rather than mean.

**2 — Establish the reading budget.** How many items can one reviewer consider
properly in a shift? Ask them; the number is usually between 20 and 30 and it
is always far below the queue.

**3 — Model detection against position.** Place a malicious item at various
depths and compute the probability it is actually read. The curve falls off a
cliff at the reading budget, not gradually.

**4 — Note who controls position.** If a requester can generate the items in
front of theirs, depth is attacker-chosen and the average case is irrelevant.

**5 — Report the two numbers that change the design.** Items per reviewer per
day, and the reading budget. The gap between them is the finding, and it points
at routing by reversibility rather than at hiring.

## Example

**Input** — the fixture committed at the top of [`scripts/approval_queue_saturation_model.py`](scripts/approval_queue_saturation_model.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
 daily volume  considered  stamped  caught  missed
           10          10        0       1       0
           25          25        0       1       0
          100          25       75       0       1
          500          25      475       0       1

At every volume the audit trail shows a human approval on 100% of
actions. The control reports full coverage in all four rows.
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "volume": {"per_reviewer_per_day": 0, "measured_at": "peak|mean"},
  "reading_budget": 0,
  "coverage_reported": 1.0,
  "detection_by_depth": [{"depth": 0, "read_probability": 0.0}],
  "position_attacker_controlled": true,
  "gap": 0
}
```

## Failure modes

- **Reporting coverage.** It is 100% by construction and means nothing.
- **Using mean volume.** The gate fails at peak.
- **Recommending more reviewers.** The fix is fewer items, chosen by
  reversibility.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/approval-queue-saturation-model/scripts/approval_queue_saturation_model.py
SCRIPT = "skills/threats/approval-queue-saturation-model/scripts/approval_queue_saturation_model.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Approval coverage reads 100% at every volume while the malicious request is caught only when the queue is small enough to be read — and an attacker choosing the position needs only to generate the requests in front of it.

## Your turn

Count how many approval requests one of your agents generates per day and ask the person approving them how many they read in full. The gap between those two numbers is the control's real coverage.

---

**Next → [A1.16 · Misaligned and deceptive behaviour](https://spbreed.github.io/cyber-commons/lessons/A1.16.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.15.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.15.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*